## 2.4. 微积分
在深度学习中，我们“训练”模型，不断更新它们，使它们在看到越来越多的数据时变得越来越好。 通常情况下，变得更好意味着最小化一个损失函数（loss function）， 即一个衡量“模型有多糟糕”这个问题的分数。 最终，我们真正关心的是生成一个模型，它能够在从未见过的数据上表现良好。 但“训练”模型只能将模型与我们实际能看到的数据相拟合。 因此，我们可以将拟合模型的任务分解为两个关键问题：

- 优化（optimization）：用模型拟合观测数据的过程；

- 泛化（generalization）：数学原理和实践者的智慧，能够指导我们生成出有效性超出用于训练的数据集本身的模型。


### 2.4.1 导数和微分

我们首先讨论导数的计算，这是几乎所有深度学习优化算法的关键步骤。
在深度学习中，我们通常选择对于模型参数可微的损失函数。
简而言之，对于每个参数，
如果我们把这个参数*增加*或*减少*一个无穷小的量，可以知道损失会以多快的速度增加或减少，

假设我们有一个函数$f: \mathbb{R} \rightarrow \mathbb{R}$，其输入和输出都是标量。
(**如果$f$的*导数*存在，这个极限被定义为**)

(**$$f'(x) = \lim_{h \rightarrow 0} \frac{f(x+h) - f(x)}{h}.$$**)


如果$f'(a)$存在，则称$f$在$a$处是*可微*（differentiable）的。
如果$f$在一个区间内的每个数上都是可微的，则此函数在此区间中是可微的。
我们可以将导数$f'(x)$解释为$f(x)$相对于$x$的*瞬时*（instantaneous）变化率。
所谓的瞬时变化率是基于$x$中的变化$h$，且$h$接近$0$。



### 2.4.2梯度
#### 第一部分：理解导数与梯度的关系

##### 1. 什么是求导 (Derivative)？
*   **适用对象**：一元函数（一个输入，一个输出），例如 $y = x^2$。
*   **物理意义**：曲线在某一点的**切线斜率**。
*   **大白话**：当你把 $x$ 稍微改变一点点时，$y$ 会跟着改变多少。它是“变化率”。

##### 2. 什么是梯度 (Gradient)？
*   **适用对象**：多元函数（多个输入，一个输出），例如标量函数 $y = f(x_1, x_2, x_3)$。
*   **定义**：梯度是一个**向量**，由函数对各个变量的“偏导数”组合而成。
    $$ \nabla y = \left[ \frac{\partial y}{\partial x_1}, \frac{\partial y}{\partial x_2}, \frac{\partial y}{\partial x_3} \right]^T $$
*   **几何意义（核心考点）**：
    *   梯度是一个方向！它指向的是函数值**增加最快**的方向。
    *   深度学习为什么叫“梯度下降”？**因为我们的目标是让损失函数（误差）变小，既然梯度指向上升最快的方向，那么逆着梯度的方向**（负梯度方向）走，就是下降最快的方向！

#### 3. 求导与梯度的关系总结
*   **导数是梯度的特例**：在一维空间里，梯度退化成普通的导数。
*   **梯度是导数的高维拓展**：导数只能告诉你“往左走还是往右走”，而梯度能告诉你“在三维甚至上万维的空间里，往哪个确切的角度走最陡”。

---

#### 第二部分：向量可以对标量求导吗？
**答案是：当然可以！**

*   **场景**：输入一个**标量 $x$**，输出一个**向量 $\mathbf{y}$**。
    假设 $\mathbf{y} = [y_1, y_2, y_3]^T$。
*   **结果是什么？**
    向量对标量求导，结果依然是一个**与原向量同维度的向量**。只需要把向量里的每一个元素单独对标量 $x$ 求导即可。
    $$ \frac{\partial \mathbf{y}}{\partial x} = \left[ \frac{\partial y_1}{\partial x}, \frac{\partial y_2}{\partial x}, \frac{\partial y_3}{\partial x} \right]^T $$
*   **现实物理例子**：
    假设标量 $t$ 代表时间，向量 $\mathbf{p}(t) = [x(t), y(t), z(t)]^T$ 代表一架飞机在三维空间中的位置。
    把位置向量 $\mathbf{p}$ 对时间标量 $t$ 求导，得到的就是**速度向量** $\mathbf{v} = [v_x, v_y, v_z]^T$。

---

#### 第三部分：向量对向量的求导是什么样的？

*   **场景**：输入一个由 $n$ 个元素组成的向量 $\mathbf{x}$，经过某个函数，输出一个由 $m$ 个元素组成的向量 $\mathbf{y}$。
*   **结果是什么？**
    一个向量对另一个向量求导，结果是一个 **矩阵**！这个矩阵在数学上有一个赫赫有名的名字——**雅可比矩阵 (Jacobian Matrix)**。
*   **具体构造**：
    如果 $\mathbf{y}$ 有 $m$ 个元素，$\mathbf{x}$ 有 $n$ 个元素，那么求导结果是一个 $m \times n$ 的矩阵（假设采用分子布局）。
    矩阵里的第 $i$ 行、第 $j$ 列的元素，就是输出向量的第 $i$ 个元素对输入向量的第 $j$ 个元素求偏导。
    $$ J = \frac{\partial \mathbf{y}}{\partial \mathbf{x}} = 
    \begin{bmatrix}
    \frac{\partial y_1}{\partial x_1} & \frac{\partial y_1}{\partial x_2} & \cdots & \frac{\partial y_1}{\partial x_n} \\
    \frac{\partial y_2}{\partial x_1} & \frac{\partial y_2}{\partial x_2} & \cdots & \frac{\partial y_2}{\partial x_n} \\
    \vdots & \vdots & \ddots & \vdots \\
    \frac{\partial y_m}{\partial x_1} & \frac{\partial y_m}{\partial x_2} & \cdots & \frac{\partial y_m}{\partial x_n}
    \end{bmatrix}
    $$
*   **深度学习中的应用**：在神经网络的反向传播（链式法则）中，相邻两层之间信息的传递（比如隐藏层 $\mathbf{h}$ 对输入层 $\mathbf{x}$ 求导），本质上就是雅可比矩阵的乘法。

---

#### 第四部分：矩阵怎么求导？（深度学习必杀技）

在纯数学中，矩阵对矩阵求导会产生四维张量，极其复杂。但**好消息是，在深度学习中你几乎永远不需要做“矩阵对矩阵”的求导！**

在深度学习中，我们永远在做一件事：**标量对矩阵/向量求导**。
*   **为什么？** 因为无论你的网络输出是什么，最终我们都会计算出一个**损失函数 (Loss)**。Loss 是一个实实在在的**标量（一个具体的数字，代表误差大小）**。我们需要求 Loss 对网络权重矩阵 $W$ 的导数，来更新 $W$。

##### 💡 深度学习矩阵求导的“第一铁律”：形状一致原则
**标量 $L$ 对矩阵 $W$ 求导，得到的梯度矩阵的形状（Shape），必定与矩阵 $W$ 的形状一模一样！**
*   如果 $W$ 是一个 $3 \times 4$ 的矩阵，那么 $\frac{\partial L}{\partial W}$ 也绝对是一个 $3 \times 4$ 的矩阵。
*   梯度矩阵中的每一个位置上的值，就是 Loss 对 $W$ 中对应位置参数的偏导数。

##### 实战例子：线性层求导
假设一层极其简单的网络，输入向量 $\mathbf{x}$（列向量 $n \times 1$），权重矩阵 $W$ ($m \times n$)，输出向量 $\mathbf{y}$（列向量 $m \times 1$）：
$$ \mathbf{y} = W \mathbf{x} $$
最终由 $\mathbf{y}$ 计算出了一个标量误差 $L$。假设我们已经求出了 $L$ 对 $\mathbf{y}$ 的梯度 $\frac{\partial L}{\partial \mathbf{y}}$ (形状同 $\mathbf{y}$ 为 $m \times 1$)。

现在求 $\frac{\partial L}{\partial W}$，直接使用**维度相容推导法（凑形状）**：
1. 链式法则告诉我们它和 $\frac{\partial L}{\partial \mathbf{y}}$ 还有 $\mathbf{x}$ 有关。
2. 目标形状：$\frac{\partial L}{\partial W}$ 必须是 $m \times n$。
3. 已知材料：$\frac{\partial L}{\partial \mathbf{y}}$ 是 $m \times 1$，$\mathbf{x}$ 是 $n \times 1$。
4. 怎么把 $m \times 1$ 和 $n \times 1$ 乘出一个 $m \times n$ 的矩阵？
   只有一种组合方式：**前者乘以结合后者的转置！**
$$ \frac{\partial L}{\partial W} = \frac{\partial L}{\partial \mathbf{y}} \mathbf{x}^T $$
*验证： $(m \times 1) \times (1 \times n) = (m \times n)$，完美符合！*

---

#### 🌟 终极核心速查表

在深度学习中，你只需要牢记这张对应表（假设 $L$ 为标量 Loss）：

| 求导类型 | 符号表示 | 结果的数据结构（形状 Shape） | 物理/数学意义 |
| :--- | :--- | :--- | :--- |
| **标量对标量** | $\frac{\partial y}{\partial x}$ | **标量** (1个数字) | 基础导数 (切线斜率) |
| **向量对标量** | $\frac{\partial \mathbf{y}}{\partial x}$ | **向量** (与 $\mathbf{y}$ 形状相同) | 比如：位置对时间求导=速度 |
| **标量对向量** | $\frac{\partial L}{\partial \mathbf{x}}$ | **向量** (与 $\mathbf{x}转置$ 形状相同) | **梯度**，指向最快上升方向 |
| **向量对向量** | $\frac{\partial \mathbf{y}}{\partial \mathbf{x}}$ | **矩阵** (行数为 $\mathbf{y}$长，列数为 $\mathbf{x}$长) | **雅可比矩阵**，反向传播的桥梁 |
| **标量对矩阵** | $\frac{\partial L}{\partial W}$ | **矩阵** (与 $W$ 形状完全相同) | **权重梯度矩阵**，用于更新网络参数 |

#### 两个常见的求导
这两个公式其实是基于我们在上一轮提到的“标量对向量求导得到行向量”这一维度对齐法则（即分子布局/Numerator-layout）。

我们把它们拆开来看，本质上就是把我们熟悉的高中求导法则，扩展到了向量空间。

##### 1. 平方 $L_2$ 范数的求导：$y = \|\mathbf{x}\|^2$

在图表中，$y = \|\mathbf{x}\|^2$ 求导结果是 $2\mathbf{x}^T$。这就相当于标量求导中的 $y = x^2$ 导数是 $2x$。

**推导过程：**
一个向量 $\mathbf{x} = [x_1, x_2, \dots, x_n]^T$ 的平方 $L_2$ 范数，本质上就是它自己和自己的内积（点乘），即所有元素的平方和：


$$y = \|\mathbf{x}\|^2 = \mathbf{x}^T\mathbf{x} = x_1^2 + x_2^2 + \dots + x_n^2$$

我们要求的是标量 $y$ 对列向量 $\mathbf{x}$ 的导数 $\frac{\partial y}{\partial \mathbf{x}}$。根据约定，结果应该是一个**行向量**。
我们先对 $\mathbf{x}$ 中的某一个特定元素 $x_i$ 求偏导：


$$\frac{\partial y}{\partial x_i} = \frac{\partial}{\partial x_i}(x_1^2 + x_2^2 + \dots + x_n^2) = 2x_i$$

把所有的偏导数组装成一个行向量：


$$\frac{\partial y}{\partial \mathbf{x}} = \left[ \frac{\partial y}{\partial x_1}, \frac{\partial y}{\partial x_2}, \dots, \frac{\partial y}{\partial x_n} \right] = [2x_1, 2x_2, \dots, 2x_n]$$

提出常数 2，剩下的正好是原向量 $\mathbf{x}$ 的转置：


$$\frac{\partial y}{\partial \mathbf{x}} = 2[x_1, x_2, \dots, x_n] = 2\mathbf{x}^T$$

---

##### 2. 内积的求导：$y = \langle\mathbf{u}, \mathbf{v}\rangle$

$y = \langle\mathbf{u}, \mathbf{v}\rangle$ 的导数是 $\mathbf{u}^T\frac{\partial \mathbf{v}}{\partial \mathbf{x}} + \mathbf{v}^T\frac{\partial \mathbf{u}}{\partial \mathbf{x}}$。这其实就是标量乘法求导法则（乘积法则） $(uv)' = u'v + uv'$ 在矩阵里的翻版。

**推导过程：**
这里的前提是，向量 $\mathbf{u}$ 和 $\mathbf{v}$ 都是关于向量 $\mathbf{x}$ 的函数。内积可以写成转置相乘的形式：


$$y = \langle\mathbf{u}, \mathbf{v}\rangle = \mathbf{u}^T\mathbf{v}$$

按照乘积法则，我们分两步来求（保持一项为常数，对另一项求导）：

**第一步：假设 $\mathbf{u}$ 是常数，对 $\mathbf{v}$ 求导**
根据链式法则，$\mathbf{u}^T\mathbf{v}$ 对 $\mathbf{x}$ 的导数是 $\mathbf{u}^T$ 乘以 $\mathbf{v}$ 对 $\mathbf{x}$ 的雅可比矩阵：


$$\mathbf{u}^T \frac{\partial \mathbf{v}}{\partial \mathbf{x}}$$

**第二步：假设 $\mathbf{v}$ 是常数，对 $\mathbf{u}$ 求导**
因为内积满足交换律，$\mathbf{u}^T\mathbf{v} = \mathbf{v}^T\mathbf{u}$。现在我们把 $\mathbf{v}$ 当作常数，对 $\mathbf{u}$ 求导，同理可得：


$$\mathbf{v}^T \frac{\partial \mathbf{u}}{\partial \mathbf{x}}$$

**第三步：将两部分相加**
把上面两项加起来，就得到了图表中的完整公式：


$$\frac{\partial y}{\partial \mathbf{x}} = \mathbf{u}^T \frac{\partial \mathbf{v}}{\partial \mathbf{x}} + \mathbf{v}^T \frac{\partial \mathbf{u}}{\partial \mathbf{x}}$$

**维度验证：**
假设 $\mathbf{x}$ 长度为 $n$，$\mathbf{u}$ 和 $\mathbf{v}$ 长度为 $m$。

* $\mathbf{u}^T$ 是 $1 \times m$ 的行向量。
* $\frac{\partial \mathbf{v}}{\partial \mathbf{x}}$ 是 $m \times n$ 的雅可比矩阵。
* 它们相乘得到 $1 \times n$ 的行向量。
两项加起来依然是 $1 \times n$ 的行向量，完美符合“标量 $y$ 对列向量 $\mathbf{x}$ 求导”的维度要求。

#### 链式法则
![链式法则示意图](img/链式法则1.png) 
![正向与反向](img/自动求导.png) 
